### 문제 3-1 :  콘텐츠분쟁해결 RAG 시스템 - 간단 실습 가이드



#### 문제 설명
콘텐츠분쟁해결 사례집을 활용한 법률 자문 RAG 시스템을 순차적으로 구축하는 실습입니다.
콘텐츠분쟁해결_사례.pdf 문서는 data\ 폴더 아래에 있습니다.


- 키 설정

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# ChatUpstage, UpstageEmbeddings 가 UPSTAGE_API_KEY 환경 변수를 자동으로 읽음 (키 값은 출력하지 않음)
if not os.getenv("UPSTAGE_API_KEY"):
    raise RuntimeError(".env 파일에 UPSTAGE_API_KEY 가 없습니다.")
print("UPSTAGE_API_KEY 설정 확인 완료")

- 0단계 : 문서 로드

loader = PyPDFLoader('../data/콘텐츠분쟁해결_사례.pdf')


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate

from langchain_upstage import ChatUpstage
from langchain_upstage import UpstageEmbeddings

print("==> 0. 문서 로딩 → PDF 읽기...")
PDF_PATH = "../data/콘텐츠분쟁해결_사례.pdf"
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()
print(documents)
print(f"  총 {len(documents)}페이지 로드 완료")

- 1단계: 문서 분할 설정 

법률 사례 1200~1800자 권장

맥락 보존 200~400자

구분자 

In [ ]:
print("==> 1. 문서 분할 → 작은 청크로 나누기")
#문서 작은 청크로 나누기
#어떤 문서. 
#작은 청크? 사이즈? 맥락보존을 위해 겹치는 부분 고려. 어느 부분에서 자를지?
#나누기
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,        # 청크 크기 
    chunk_overlap=300,      # 중복 부분 (맥락 보존)
    separators=["\n\n", "\n", ".", " ", ""] # 자연스러운 분할을 위한 구분자
)

chunks = text_splitter.split_documents(documents)
#print(chunks)
print(f"  {len(chunks)}개 청크 생성 완료")
print(f"  평균 청크 길이: {sum(len(chunk.page_content) for chunk in chunks) / len(chunks):.0f}자")
print(type(chunks[0]))

- 2단계: 임베딩 모델 설정 

pythonembeddings = UpstageEmbeddings(model="solar-embedding-1-large")

In [ ]:
print("==> 2. 벡터화 → 임베딩으로 변환")
pythonembeddings = UpstageEmbeddings(model="solar-embedding-1-large")

print("==> 저장 → FAISS 벡터스토어에 저장")
vectorstore = FAISS.from_documents(chunks, pythonembeddings)
print(f" FAISS 벡터스토어 생성 완료 ({len(chunks)}개 벡터)")

#로컬 파일 저장
DB_PATH = "../db/faiss_db_upstage"
vectorstore.save_local(DB_PATH)
print(f" FAISS 인덱스 저장 완료: {DB_PATH}")

- 3단계: 검색기 설정 

pythonretriever = vectorstore.as_retriever()

In [ ]:
print("===> 3. 검색 → 질문과 유사한 문서 찾기")
pythonretriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}  # 상위 5개 관련 문서 검색
)
print(" Retriever 설정 완료")

- 4단계: LLM 설정 

pythonllm = ChatUpstage(
    
)

In [ ]:
print("===> 4. 생성 → LLM으로 답변 생성")
pythonllm = ChatUpstage(
        model="solar-pro3",
        base_url="https://api.upstage.ai/v1",
        temperature=0.2
)
print(pythonllm.model_name)

- 5단계: 법률 자문 프롬프트 작성 

pythonprompt_template = """
당신은 콘텐츠 분야 전문 법률 자문가입니다. 
아래 분쟁조정 사례들을 바탕으로 정확하고 전문적인 법률 조언을 제공해주세요.

관련 분쟁사례:
{context}

상담 내용: {question}

답변 가이드라인:
1. 제시된 사례들을 근거로 답변하세요                    # 사례 기반 답변
2. 관련 법령이나 조항이 있다면 명시하세요               # 법적 근거 제시
3. 비슷한 사례의 처리경위와 결과를 참고하여 설명하세요    # 판례 참조
4. 실무적 해결방안을 단계별로 제시하세요               #  실무 가이드
5. 사례에 없는 내용은 "제시된 사례집에서는 확인할 수 없습니다"라고 명시하세요  #  한계 인정

전문 법률 조언:"""


In [ ]:
pythonprompt_template = """
당신은 콘텐츠 분야 전문 법률 자문가입니다. 
아래 분쟁조정 사례들을 바탕으로 정확하고 전문적인 법률 조언을 제공해주세요.

관련 분쟁사례:
{context}

상담 내용: {question}

답변 가이드라인:
1. 제시된 사례들을 근거로 답변하세요                    # 사례 기반 답변
2. 관련 법령이나 조항이 있다면 명시하세요               # 법적 근거 제시
3. 비슷한 사례의 처리경위와 결과를 참고하여 설명하세요    # 판례 참조
4. 실무적 해결방안을 단계별로 제시하세요               #  실무 가이드
5. 사례에 없는 내용은 "제시된 사례집에서는 확인할 수 없습니다"라고 명시하세요  #  한계 인정

전문 법률 조언:"""

prompt = PromptTemplate.from_template(pythonprompt_template)
print(prompt)


def format_docs(docs):
    """문서들을 문자열로 포맷"""
    return "\n\n".join(doc.page_content for doc in docs)


- 6단계: LCEL 체인 생성 

pythonqa_chain = (
    
)


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

pythonqa_chain = (
    {"context": pythonretriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | pythonllm
    | StrOutputParser()
)

print("  LCEL RAG 파이프라인 구축 완료!")

- 7단계: 테스트 질문 작성과 테스트

pythontest_questions = [

    "온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?",
    "인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 요구받고 있습니다. 정당한가요?",
    "무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다. 환불 가능한가요?",
    "미성년자가 부모 동의 없이 게임 아이템을 구매했습니다. 환불받을 수 있는 방법이 있나요?",
    "온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해지하고 싶습니다. 가능한가요?"

]


In [ ]:
pythontest_questions = [
    "온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?",
    "인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 요구받고 있습니다. 정당한가요?",
    "무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다. 환불 가능한가요?",
    "미성년자가 부모 동의 없이 게임 아이템을 구매했습니다. 환불받을 수 있는 방법이 있나요?",
    "온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해지하고 싶습니다. 가능한가요?"
]

#답변과 출처를 같이 
def rag_with_sources(question):
    #유사도 검색
    docs = pythonretriever.invoke(question)
    context = format_docs(docs)
    
    response = pythonllm.invoke(prompt.format(context=context, question=question))
    return {
        "answer": response.content,
        "source_documents": docs
    }


#테스트
print("\n" + "=" * 60)
print(" LCEL RAG 시스템 테스트")
print("=" * 60)

for i, question in enumerate(pythontest_questions, 1):
    print(f"\n【테스트 {i}/5】")
    print(f"Q: {question}")
    
    result = rag_with_sources(question)
    print(f"A: {result['answer']}")

    docs = result["source_documents"]

    print(f"  참조: {len(docs)}개 문서")
    for j, doc in enumerate(docs[:2], 1):
        page = doc.metadata.get('page_label', '?')
        print(f"    {j}. P{page}: {doc.page_content[:80]}...")
    
    print("-" * 40)